# Ames Housing Price Prediction
## 1. Project Overview

**Project Title**: Ames Housing Price Prediction
**Description**: Machine learning project for predicting residential property prices using the Ames Housing dataset, with regression model comparison and property valuation feature analysis.
**Business Scenario**: A real estate company wants to understand property value factors and support better pricing decisions.

### Lenses
*   **Primary Lens (Price Prediction)**: Can machine learning predict the expected sale price of a residential property from its characteristics?
*   **Secondary Lens (Valuation Feature Analysis)**: Which property characteristics are most influential in predicting residential property prices?

**ML Task**: Supervised Regression
**Target Variable**: `SalePrice`


## 2. Import Libraries

Here we import all the necessary libraries for data manipulation, visualization, and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Set reproducible seed
np.random.seed(42)

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## 3. Load Dataset

We will load the training dataset which contains the target variable `SalePrice`.


In [ ]:
# Load dataset
df = pd.read_csv('../data/train.csv')

print(f"Dataset Shape: {df.shape}\n")
print("First 5 rows:")
display(df.head())

print("\nDataset Info:")
df.info()


## 4. Data Understanding

In this section, we investigate the dimensions, data types, and missing values.


In [ ]:
# Basic summary statistics for numerical columns
display(df.describe())

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Identify missing values
missing_counts = df.isnull().sum()
missing_vars = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("\nMissing values per column:")
display(missing_vars)


## 5. Exploratory Data Analysis (EDA)

The EDA focuses on understanding the distribution of `SalePrice` and its relationship with key features.


In [ ]:
# Target variable distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True)
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')
plt.show()

print(f"Skewness: {df['SalePrice'].skew():.4f}")
print(f"Mean: {df['SalePrice'].mean():.2f}")
print(f"Median: {df['SalePrice'].median():.2f}")


In [ ]:
# Relationships with key numerical features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x='GrLivArea', y='SalePrice', data=df, ax=axes[0])
axes[0].set_title('SalePrice vs GrLivArea')

sns.scatterplot(x='TotalBsmtSF', y='SalePrice', data=df, ax=axes[1])
axes[1].set_title('SalePrice vs TotalBsmtSF')

plt.tight_layout()
plt.show()


In [ ]:
# Relationships with categorical features
plt.figure(figsize=(12, 6))
sns.boxplot(x='OverallQual', y='SalePrice', data=df)
plt.title('SalePrice vs OverallQual')
plt.show()


In [ ]:
# Correlation heatmap for top numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

# Get top 10 features correlated with SalePrice
top_corr_features = corr_matrix.nlargest(11, 'SalePrice')['SalePrice'].index

plt.figure(figsize=(10, 8))
sns.heatmap(df[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Top Features with SalePrice')
plt.show()


## 6. Outlier Analysis

We observed some potential outliers in the `GrLivArea` plot (very large houses with relatively low prices). Let's investigate.


In [ ]:
# Identifying outliers
outliers = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)]
display(outliers)

# Removing outliers as they might negatively affect linear models and are not representative
df = df.drop(outliers.index)
print(f"Dataset shape after outlier removal: {df.shape}")


## 7. Feature Engineering

We create new features that might be more predictive of the target variable.


In [ ]:
# Total area feature
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# Property Age
df['PropertyAge'] = df['YrSold'] - df['YearBuilt']

# Total Bathrooms
df['TotalBaths'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
